# Aula 1 — Kick-off: o que vamos construir e por que

**Intensivão Quant AI — ImpactUFSCar**

---

Bem-vindo ao intensivão. Ao longo de 5 semanas, você vai construir do zero uma estratégia quantitativa real — do mesmo tipo que equipes submetem ao Desafio Quant AI da Itaú Asset Management.

Esta primeira aula não tem muito código. O objetivo é alinhar todo mundo em três coisas:
1. O que é uma estratégia quantitativa (e o que não é)
2. O que a banca do desafio avalia e como
3. O que vamos construir juntos

In [ ]:
# ── CONFIGURAÇÃO DO AMBIENTE ─────────────────────────────────────────────────
# Este notebook roda no VS Code (Jupyter local) E no Google Colab.
# Execute esta célula primeiro — ela detecta o ambiente e configura os caminhos.

import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # ── Google Colab ──────────────────────────────────────────────────────────
    print("Ambiente detectado: Google Colab")

    # Montar Google Drive para persistir os dados entre sessões
    from google.colab import drive
    drive.mount('/content/drive')

    # Instalar dependências não incluídas no Colab por padrão
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'yfinance', 'pyarrow'], check=False)

    # Clonar o repositório do curso (pula automaticamente se já existir)
    REPO_DIR = '/content/intensivao-quant-ai'
    if not os.path.exists(REPO_DIR):
        print("Clonando repositório do curso...")
        subprocess.run([
            'git', 'clone',
            'https://github.com/fmaldonadoo/intensivao-quant-ai.git',
            REPO_DIR
        ])
        print("Repositório clonado com sucesso.")

    # Pasta de dados no Google Drive — persiste entre sessões do Colab
    DADOS_DIR = '/content/drive/MyDrive/intensivao_quant/dados'
    os.makedirs(DADOS_DIR, exist_ok=True)
    print(f"Dados serão salvos em: {DADOS_DIR}")

else:
    # ── VS Code / Jupyter local ───────────────────────────────────────────────
    print("Ambiente detectado: VS Code / Jupyter local")

    # Sobe a árvore de diretórios até encontrar a raiz do repositório (.git)
    _p = os.path.abspath(os.getcwd())
    while _p != os.path.dirname(_p):
        if os.path.exists(os.path.join(_p, '.git')):
            break
        _p = os.path.dirname(_p)

    DADOS_DIR = os.path.join(_p, 'dados')
    os.makedirs(DADOS_DIR, exist_ok=True)
    print(f"Dados serão salvos em: {DADOS_DIR}")


## 1. O que é uma estratégia quantitativa

Uma estratégia quantitativa — ou **robô de investimento** — é um conjunto de regras objetivas e replicáveis que decide:

- **O quê** comprar ou vender
- **Quando** comprar ou vender
- **Quanto** alocar em cada ativo

A palavra-chave é **objetivas**: as regras não dependem de julgamento humano no momento da decisão. Dado o mesmo conjunto de dados, dois computadores diferentes chegariam à mesma conclusão.

### O que não é uma estratégia quantitativa

- "Compro quando o gráfico parece bom" — subjetivo
- "Sigo a recomendação do analista" — não é replicável
- "Compro ações de empresas que acho que vão crescer" — sem regra explícita

### A estrutura de qualquer estratégia quant

```
DADOS  →  SINAL  →  ALOCAÇÃO  →  EXECUÇÃO
```

- **Dados:** preços, volumes, fundamentos, notícias — a matéria-prima
- **Sinal:** uma medida calculada a partir dos dados que indica quais ativos favorecer
- **Alocação:** como traduzir o sinal em pesos de carteira (quanto em cada ativo)
- **Execução:** como comprar/vender na prática (custos, liquidez, frequência)

## 2. O Desafio Quant AI do Itaú

O Desafio Quant AI é a maior competição de finanças quantitativas do Brasil. Cada equipe submete um relatório apresentando sua estratégia.

### O que a banca avalia

| Critério | Peso | O que significa na prática |
|---|---|---|
| Conceito da Estratégia | 20% | A tese econômica faz sentido? Há uma razão para funcionar? |
| Modelagem | 20% | O sinal é construído com rigor? A lógica está correta? |
| Uso de IA Generativa | 15% | GenAI foi usada de forma prática no processo? |
| Backtest | 15% | O teste histórico é honesto? Tem custos? Sem look-ahead? |
| Análise dos Resultados | 15% | Os resultados são interpretados com profundidade? |
| Conclusão e Próximos Passos | 10% | A equipe entende os limites da própria estratégia? |
| Apresentação do Robô | 5% | Nome, identidade, clareza da apresentação |

### O que a banca não é

A banca **não** premia o maior retorno. Uma estratégia que rendeu 15% ao ano modelada com rigor vence uma que rendeu 40% com backtest mal-feito.

Isso importa: **você não precisa descobrir o Santo Graal**. Você precisa fazer um trabalho sólido.

## 3. O que separa um backtest honesto de um ruim

Este é o conceito mais importante desta aula. A maioria dos backtests na internet é enganosa — não por má-fé, mas por erros sutis que inflam os resultados.

### Look-ahead bias

O erro mais comum. Ocorre quando você usa informação do futuro para tomar uma decisão do passado.

**Exemplo:** em janeiro de 2020, você "decide" comprar ações que vão se recuperar bem da pandemia. Mas em janeiro de 2020 você não sabia da pandemia.

No código, esse erro aparece assim:
```python
# ERRADO — usa retorno do próprio mês no ranking
sinal = retorno_janeiro
carteira_janeiro = top_N(sinal)  # você "sabia" em 01/jan o retorno de todo janeiro

# CORRETO — usa retorno até o mês anterior
sinal = retorno_dezembro
carteira_janeiro = top_N(sinal)  # você sabia isso no início de janeiro
```

### Overfitting

Se você testa 50 variações da estratégia e publica só a que funcionou melhor, não encontrou uma boa estratégia — encontrou ruído que parece sinal.

### Custos ignorados

Toda compra e venda tem custo: spread, corretagem, impacto de mercado. Uma estratégia que rotaciona a carteira toda semana pode parecer ótima bruta e terrível líquida.

Vamos atacar cada um desses erros ao longo do intensivão. Você vai aprender a identificá-los — e a banca também sabe identificá-los.

## 4. Nossa estratégia: Momentum Cross-Seccional no IBOVESPA

### A tese

**Ações do IBOVESPA que tiveram os maiores retornos nos últimos 12 meses tendem a continuar superando as demais nos próximos meses.**

Isso é **momentum**. É um dos fenômenos mais estudados em finanças — Jegadeesh e Titman documentaram isso nos EUA em 1993, e o efeito foi replicado em dezenas de mercados, incluindo o Brasil.

### Por que funciona (a intuição)

Investidores demoram a incorporar informação nova. Quando uma empresa publica um resultado melhor do que o esperado, o preço sobe — mas devagar, porque muitos investidores ainda não ajustaram suas expectativas. Esse ajuste lento cria persistência: o que subiu tende a continuar subindo por alguns meses.

### O que vamos construir

```
Aula 2  → Baixar preços de todas as ações do IBOVESPA
Aula 3  → Explorar os dados, entender o que temos
Aula 4  → Calcular o sinal de momentum, montar a carteira
Aula 5  → Testar no histórico, medir Sharpe e drawdown
Aula 6  → Melhorar a alocação
Aula 7  → Refinar o sinal
Aula 8  → Fazer o backtest de forma honesta
Aula 9  → Analisar os resultados com ajuda de IA
Aula 10 → Montar o relatório e defender a estratégia
```

### Uma prévia do produto final

Na Aula 5 você vai gerar um gráfico como este:

```
Retorno acumulado (2010–2024)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Momentum top 10:   +XXX%   Sharpe: X.XX
IBOVESPA:          +XXX%   Sharpe: X.XX
```

Se a estratégia funcionar, a linha do momentum vai ficar acima do IBOVESPA na maior parte do tempo. Se não funcionar ou funcionar mal, você vai entender por quê — e isso também conta na avaliação.

## 5. Setup para a próxima aula

Antes da Aula 2, você precisa ter o ambiente pronto. Faça isso em casa.

### Opção A — Google Colab (recomendada se for sua primeira vez)

1. Acesse [colab.research.google.com](https://colab.research.google.com)
2. Crie um novo notebook
3. Rode a célula abaixo para instalar as dependências

In [ ]:
# Rode esta célula no Colab para instalar as dependências
!pip install yfinance pandas numpy matplotlib statsmodels -q
print("Pronto!")

### Opção B — Python local

```bash
pip install yfinance pandas numpy matplotlib statsmodels
```

### Verificação

Rode a célula abaixo. Se aparecer os números de versão, está tudo certo.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

print(f"pandas     {pd.__version__}")
print(f"numpy      {np.__version__}")
print(f"yfinance   {yf.__version__}")
print()

# Teste rápido: baixar o IBOVESPA
ibov = yf.download("^BVSP", start="2024-01-01", end="2024-01-31", progress=False)
print(f"IBOVESPA baixado: {len(ibov)} dias de dados")
print(ibov.tail(3))

Se o código acima funcionou e você viu alguns dias de dados do IBOVESPA, você está pronto para a Aula 2.

---

## Para a próxima aula

**Ambiente funcionando** (obrigatório) — rode a célula de verificação acima.

**Opcional — assistir antes:** MIT 18.S096, Lecture 1 (*Introduction*) — 20 min. Não é pré-requisito, mas dá contexto sobre o que matemática aplicada a finanças significa na prática.

---

*ImpactUFSCar — Diretoria de Quant*